# Fine-Tuned LLM Term Extraction with Unsloth

This notebook demonstrates term extraction by fine-tuning a Large Language Model:
- Uses **Gemma-3-4B** model with efficient fine-tuning via Unsloth
- Trains on DETECH dataset
- Optimizes using LoRA (Low-Rank Adaptation) for efficient training
- Evaluates with official competition metrics

Unsloth provides 2x faster training with 60% less memory usage.

<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>
</div>

## Setup and Imports

In [1]:
# Install Unsloth and dependencies
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install unsloth_zoo
!pip install bitsandbytes
print("✓ Unsloth and dependencies installed")

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["UNSLOTH_DISABLE_FAST_GENERATION"] = "1"
os.environ["UNSLOTH_DISABLE_STATIC_GENERATION"] = "1"

In [ ]:
# Get your HF Token here: https://huggingface.co/settings/tokens
import os
os.environ["HF_TOKEN"] = input("Insert your HF Token and press Enter: ")

## Load Base Model with Unsloth

Load Gemma-3-4B model with 4-bit quantization for efficient fine-tuning.

In [3]:
from unsloth import FastModel
import torch

# Load Gemma-3-4B model with 4-bit quantization
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

print("Gemma-3-4B model loaded")
print(f"  Model: {model.__class__.__name__}")
print(f"  Max sequence length: 2048")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1111 17:22:46.478000 325 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.4: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Gemma-3-4B model loaded
  Model: PeftModelForCausalLM
  Max sequence length: 2048


## Add LoRA Adapters

Configure LoRA (Low-Rank Adaptation) for parameter-efficient fine-tuning.

In [9]:
# Add LoRA adapters for efficient fine-tuning
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

print("LoRA adapters configured")
print(f"  Rank (r): 16")
print(f"  Alpha: 16")

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
The CUDA driver logged these messages, which may provide useful details:
Sticky error detected
Returning 710 (CUDA_ERROR_ASSERT) from cuStreamGetCaptureInfo_v3



## Data Loading and Processing

Load dataset and format for term extraction task.

In [ ]:
# Configure chat template for Gemma-3
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

print("Gemma-3 chat template configured")

In [ ]:
import json

# Set DATA_DIR
DATA_DIR = "/content/"

# Load training and test data
train_data = json.load(open(f"{DATA_DIR}/train.json", 'r', encoding='utf-8'))
test_data = json.load(open(f"{DATA_DIR}/test.json", 'r', encoding='utf-8'))

print(f"Training docs: {len(train_data)}")
print(f"Test docs: {len(test_data)}")
print(f"\nExample sentence:")
print(f"  Text: {test_data[list(test_data.keys())[0]]['text']}")
print(f"  Terms: {test_data[list(test_data.keys())[0]]['terms']}")

## Format Data for Term Extraction

Convert data to conversation format for fine-tuning.

In [ ]:
prompt = f"""You are an automatic term extraction agent. You will receive a biomedical text as input.
Your role is to extract terms relevant to the gut-brain interplay field of research, including gut microbiota, mental health, and neurodegenerative diseases. Output a list of terms.

Strictly adhere to the Example Output Format:

Example Output Format:
Terms: [term1; term2; term3; term4]

Instructions:
* Extract single-word and multi-word terms, including domain-relevant organism names such as Akkermansia muciniphila
* Extract domain-relevant lexical units even if they are not listed in biomedical dictionaries or ontologies
* Extract complete lexical units, including relevant modifiers, rather than their nested component terms
* Split coordinated expressions into separate terms, restoring shared words where needed: "cognitive or motor dysfunction" yields "cognitive dysfunction" and "motor dysfunction"
* Extract only terms relevant to the gut-brain interplay, ignoring other domains
* If a document contains no terms, output an empty list for that document
* You must output a single list of terms without additional commentary

***text***
"""

In [ ]:
# Convert to conversation format for fine-tuning
from datasets import Dataset

conversations = []
for id, content in train_data.items():
    text = content['text']
    terms = content['terms']

    # Format as instruction-response pairs
    terms_str = ', '.join(terms) if terms else 'none'

    conversation = [
        {"role": "user", "content": prompt.replace("***text***", text)},
        {"role": "assistant", "content": f"Terms: {terms_str}"}
    ]
    conversations.append({"conversations": conversation})

# Create dataset
dataset = Dataset.from_list(conversations)

print(f"Dataset formatted for fine-tuning")
print(f"  Total examples: {len(dataset)}")

### Example Training Sample

In [ ]:
# Show example conversation
print("Example training conversation:")
print(dataset[100]['conversations'])

## Apply Chat Template

Apply Gemma-3 chat template to format conversations for training.

In [ ]:
# Apply chat template to conversations
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        ).removeprefix('<bos>')
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

print("Chat template applied to dataset")

### Formatted Training Example

In [ ]:
# Show formatted text
print("Formatted training example:")
print(dataset[100]["text"][:300] + "...")

## Configure Training

Set up training arguments and trainer with response masking.

In [ ]:
# Configure trainer
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,  # Full training run
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("✓ Trainer configured")
print(f"  Epochs: 1")
print(f"  Batch size: 2")
print(f"  Learning rate: 2e-4")

### Enable Response-Only Training

Train only on model responses, not user inputs.

In [ ]:
# Train only on assistant responses
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

print("✓ Response-only training enabled")

In [ ]:
# Check initial memory usage
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU: {gpu_stats.name}")
print(f"Max memory: {max_memory} GB")
print(f"Reserved: {start_gpu_memory} GB")

## Train the Model

Note: Training may take several hours depending on hardware.

In [15]:
# Start training
print("="*60)
print("Starting fine-tuning...")
print("="*60)

trainer_stats = trainer.train()

print("\n" + "="*60)
print("✓ TRAINING COMPLETED!")
print("="*60)

Starting fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 540 | Num Epochs = 1 | Total steps = 68
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,802,496 of 4,329,881,968 (0.69% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.479700
20,0.545800
30,0.483000
40,0.415800
50,0.403200
60,0.388500



✓ TRAINING COMPLETED!


In [16]:
# Show training statistics
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
training_time_minutes = round(trainer_stats.metrics['train_runtime']/60, 2)

print(f"Training time: {training_time_minutes} minutes")
print(f"Peak memory: {used_memory} GB")
print(f"Memory for training: {used_memory_for_lora} GB")

Training time: 13.88 minutes
Peak memory: 5.568 GB
Memory for training: 1.134 GB


## Save Fine-Tuned Model

In [17]:
# Save LoRA adapters
model.save_pretrained("models/gemma3_term_extraction")
tokenizer.save_pretrained("models/gemma3_term_extraction")

print("Model saved to models/gemma3_term_extraction")

✓ Model saved to models/gemma3_term_extraction


In [ ]:
!zip -r models.zip models/

## Inference and Evaluation

Run predictions on dev set and evaluate with official metrics.

In [4]:
!unzip gemma3_term_extraction.zip

Archive:  gemma3_term_extraction.zip
   creating: gemma3_term_extraction/
  inflating: gemma3_term_extraction/adapter_config.json  
  inflating: gemma3_term_extraction/adapter_model.safetensors  
  inflating: gemma3_term_extraction/added_tokens.json  
  inflating: gemma3_term_extraction/chat_template.jinja  
  inflating: gemma3_term_extraction/preprocessor_config.json  
  inflating: gemma3_term_extraction/processor_config.json  
  inflating: gemma3_term_extraction/README.md  
  inflating: gemma3_term_extraction/special_tokens_map.json  
  inflating: gemma3_term_extraction/tokenizer.json  
  inflating: gemma3_term_extraction/tokenizer.model  
  inflating: gemma3_term_extraction/tokenizer_config.json  


In [6]:
from unsloth import FastModel
import torch
from unsloth.chat_templates import get_chat_template

# Load Gemma-3-4B model with 4-bit quantization
model, tokenizer = FastModel.from_pretrained(
    #model_name = "unsloth/gemma-3-4b-it",
    model_name = "gemma3_term_extraction",
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
FastModel.for_inference(model)

print("Gemma-3-4B model loaded")
print(f"  Model: {model.__class__.__name__}")
print(f"  Max sequence length: 2048")

==((====))==  Unsloth 2026.11.4: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Gemma-3-4B model loaded
  Model: PeftModelForCausalLM
  Max sequence length: 2048


In [7]:
import json

# Set DATA_DIR
DATA_DIR = "/content/"

test_data = json.load(open(f"{DATA_DIR}/test.json", 'r', encoding='utf-8'))
MAX_DOCUMENTS = 20  # Quick first run; use None to evaluate the full test set.
if MAX_DOCUMENTS is not None:
    test_data = dict(list(test_data.items())[:MAX_DOCUMENTS])
print(f"Selected {len(test_data)} test documents")

Selected 20 test documents


## Evaluation Metrics

Using the official evaluation metrics from the competition.

In [8]:
def micro_f1_score(gold_standard, system_output):
    """
    Evaluates performance using Precision, Recall, and F1 score
    based on individual term matching (micro-average).
    """
    total_true_positives = 0
    total_false_positives = 0
    total_false_negatives = 0

    for gold, system in zip(gold_standard, system_output):
        gold_set = set(gold)
        system_set = set(system)

        true_positives = len(gold_set.intersection(system_set))
        false_positives = len(system_set - gold_set)
        false_negatives = len(gold_set - system_set)

        total_true_positives += true_positives
        total_false_positives += false_positives
        total_false_negatives += false_negatives

    precision = total_true_positives / (total_true_positives + total_false_positives) if (total_true_positives + total_false_positives) > 0 else 0
    recall = total_true_positives / (total_true_positives + total_false_negatives) if (total_true_positives + total_false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1, total_true_positives, total_false_positives, total_false_negatives


def type_f1_score(gold_standard, system_output):
    """
    Evaluates performance using Type Precision, Type Recall, and Type F1 score
    based on the set of unique terms extracted at least once across the entire dataset.
    """
    all_gold_terms = set()
    for item_terms in gold_standard:
        all_gold_terms.update(item_terms)

    all_system_terms = set()
    for item_terms in system_output:
        all_system_terms.update(item_terms)

    type_true_positives = len(all_gold_terms.intersection(all_system_terms))
    type_false_positives = len(all_system_terms - all_gold_terms)
    type_false_negatives = len(all_gold_terms - all_system_terms)

    type_precision = type_true_positives / (type_true_positives + type_false_positives) if (type_true_positives + type_false_positives) > 0 else 0
    type_recall = type_true_positives / (type_true_positives + type_false_negatives) if (type_true_positives + type_false_negatives) > 0 else 0
    type_f1 = 2 * (type_precision * type_recall) / (type_precision + type_recall) if (type_precision + type_recall) > 0 else 0

    return type_precision, type_recall, type_f1


print("Evaluation functions defined")

Evaluation functions defined


## Run Predictions on Test Set

In [9]:
prompt = f"""You are an automatic term extraction agent. You will receive a biomedical text as input.
Your role is to extract terms relevant to the gut-brain interplay field of research, including gut microbiota, mental health, and neurodegenerative diseases. Output a list of terms.

Strictly adhere to the Example Output Format:

Example Output Format:
Terms: [term1; term2; term3; term4]

Instructions:
* Extract single-word and multi-word terms, including domain-relevant organism names such as Akkermansia muciniphila
* Extract domain-relevant lexical units even if they are not listed in biomedical dictionaries or ontologies
* Extract complete lexical units, including relevant modifiers, rather than their nested component terms
* Split coordinated expressions into separate terms, restoring shared words where needed: "cognitive or motor dysfunction" yields "cognitive dysfunction" and "motor dysfunction"
* Extract only terms relevant to the gut-brain interplay, ignoring other domains
* If a document contains no terms, output an empty list for that document
* You must output a single list of terms without additional commentary

***text***
"""

In [15]:
from tqdm import tqdm

# Run inference on test set
print("Running predictions on test set...")

llm_preds = []

for id, content in tqdm(test_data.items()):
    text = content['text']
    current_prompt = prompt.replace("***text***", text)

    # Format as conversation
    conversation = [{"role": "user", "content": current_prompt}]

    # Apply chat template and tokenize
    formatted_prompt = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    # Generate
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            use_cache=False,
        )

    # Decode and parse
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract terms from "Terms: term1, term2, ..." format
    try:
        if "Terms:" in response:
            terms_part = response.split("Terms:")[-1].strip()
            if terms_part.lower() == 'none':
                terms = []
            else:
                terms = [t.strip().lower() for t in terms_part.split(',') if t.strip()]
        else:
            terms = []
    except:
        terms = []

    llm_preds.append(terms)

print(f"✓ Predictions completed: {len(llm_preds)} predictions")

Running predictions on test set...


100%|██████████| 20/20 [31:27<00:00, 94.36s/it]

✓ Predictions completed: 20 predictions


## Save Predictions to File

In [16]:
# Save predictions in competition format
def save_predictions(predictions, documents, output_path):
    """Save predictions in competition format."""
    output = {}
    idx = 0
    for id, content in documents.items():
        output[id] = {
            'text': content['text'],
            'terms': predictions[idx]
        }
        idx += 1

    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(predictions)} predictions to {output_path}")

save_predictions(llm_preds, test_data, f'predictions/subtaskA_gemma3-finetuned_preds.json')

Saved 20 predictions to predictions/subtaskA_gemma3-finetuned_preds.json


## Evaluate Performance

In [18]:
# Prepare gold standard and predictions for evaluation
test_gold = [content['terms'] for _, content in test_data.items()]

# Calculate metrics
precision, recall, f1, tp, fp, fn = micro_f1_score(test_gold, llm_preds)
type_precision, type_recall, type_f1 = type_f1_score(test_gold, llm_preds)

print("\n" + "="*60)
print("LLM ZERO-SHOT BASELINE RESULTS")
print("="*60)
print("\nMicro-averaged Metrics:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  TP={tp}, FP={fp}, FN={fn}")

print("\nType-level Metrics:")
print(f"  Type Precision: {type_precision:.4f}")
print(f"  Type Recall:    {type_recall:.4f}")
print(f"  Type F1 Score:  {type_f1:.4f}")
print("="*60)


LLM ZERO-SHOT BASELINE RESULTS

Micro-averaged Metrics:
  Precision: 0.5751
  Recall:    0.4851
  F1 Score:  0.5263
  TP=310, FP=229, FN=329

Type-level Metrics:
  Type Precision: 0.5228
  Type Recall:    0.4447
  Type F1 Score:  0.4806


## Example Predictions

In [20]:
predictions = json.load(open('predictions/subtaskA_gemma3-finetuned_preds.json', 'r', encoding='utf-8'))

# Show example predictions
print("Example Predictions:\n")

count = 0

for id, content in test_data.items():
    if len(content['terms']) > 0 and count < 5:
        print(f"Document ID: {id}")
        print(f"Text: {content['text'][:100]}...")
        print(f"Gold terms: {content['terms'][:5]}")
        print(f"LLM predictions: {predictions[id]['terms'][:5]}")

        correct = set(content['terms']) & set(predictions[id]['terms'])
        missed = set(content['terms']) - set(predictions[id]['terms'])
        wrong = set(predictions[id]['terms']) - set(content['terms'])

        print(f"Correct: {len(correct)}")
        print(f"Missed: {len(missed)}")
        print(f"Wrong: {len(wrong)}")
        print("-"*80)
        print()

        count += 1


Example Predictions:

Document ID: mh_2
Text: Gut microbiota disturbances in hospitalized older adults with malnutrition and clinical outcomes.  O...
Gold terms: ['Gut microbiota disturbances', 'malnutrition', 'clinical outcomes', 'Malnutrition', 'gut microbiota']
LLM predictions: ['gut microbiota disturbances', 'hospitalized', 'older adults', 'malnutrition', 'malnutrition']
Correct: 10
Missed: 23
Wrong: 24
--------------------------------------------------------------------------------

Document ID: mh_3
Text: Preface: Special issue: "Ion channels and genetic epilepsy".  This preface introduces the Journal of...
Gold terms: ['Ion channels', 'genetic epilepsy', 'Epilepsy', 'disease', 'recurrent seizures']
LLM predictions: ['ion channels', 'genetic epilepsy', 'journal of neurochemistry special issue', 'epilepsy', 'disease']
Correct: 17
Missed: 13
Wrong: 10
--------------------------------------------------------------------------------

Document ID: mh_5
Text: Association between gut mi